In [1]:
# ============================================================
# CELL 1 — Shared configuration (run this first)
# ============================================================
import pandas as pd
import re
from pathlib import Path
import getpass
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Current Windows user, so paths work on any machine
USER = getpass.getuser()

# Folders of FAST output CSVs
a_dir = Path(
    rf"C:\Users\{USER}\Pathways Dropbox\Projects\202501.Solano.Bayshore"
    rf"\400-Technical\450-FAST\CoastalA"
)
v_dir = Path(
    rf"C:\Users\{USER}\Pathways Dropbox\Projects\202501.Solano.Bayshore"
    rf"\400-Technical\450-FAST\CoastalV"
)

# SLR scenarios we expect to find
EXPECTED_SLR = [12, 36, 48, 52, 77, 84, 96, 108]

In [2]:
# ============================================================
# CELL 2 — Build Excel summary (3 tabs: CoastalA, CoastalV, Comparison)
# ============================================================
out_xlsx = a_dir.parent / "QA_FAST_Loss_Summary_CoastalA_vs_CoastalV.xlsx"

# --- Read totals from each folder (inline, no helper) ---
a_data = {}
print("Reading CoastalA folder...")
for p in sorted(a_dir.glob("*.csv")):
    if "sorted" in p.name.lower():
        continue
    slr = None
    for n in re.findall(r"\d+", p.name):
        if int(n) in EXPECTED_SLR:
            slr = int(n)
            break
    if slr is None:
        continue
    df = pd.read_csv(p, low_memory=False)
    a_data[slr] = {
        "BldgLoss":      df["BldgLossUSD"].sum()      if "BldgLossUSD"      in df.columns else 0,
        "ContentLoss":   df["ContentLossUSD"].sum()   if "ContentLossUSD"   in df.columns else 0,
        "InventoryLoss": df["InventoryLossUSD"].sum() if "InventoryLossUSD" in df.columns else 0,
        "file":          p.name,
    }
    print(f"  CoastalA  SLR={slr}: B=${a_data[slr]['BldgLoss']:>14,.0f}  "
          f"C=${a_data[slr]['ContentLoss']:>14,.0f}  I=${a_data[slr]['InventoryLoss']:>12,.0f}")

v_data = {}
print("\nReading CoastalV folder...")
for p in sorted(v_dir.glob("*.csv")):
    if "sorted" in p.name.lower():
        continue
    slr = None
    for n in re.findall(r"\d+", p.name):
        if int(n) in EXPECTED_SLR:
            slr = int(n)
            break
    if slr is None:
        continue
    df = pd.read_csv(p, low_memory=False)
    v_data[slr] = {
        "BldgLoss":      df["BldgLossUSD"].sum()      if "BldgLossUSD"      in df.columns else 0,
        "ContentLoss":   df["ContentLossUSD"].sum()   if "ContentLossUSD"   in df.columns else 0,
        "InventoryLoss": df["InventoryLossUSD"].sum() if "InventoryLossUSD" in df.columns else 0,
        "file":          p.name,
    }
    print(f"  CoastalV  SLR={slr}: B=${v_data[slr]['BldgLoss']:>14,.0f}  "
          f"C=${v_data[slr]['ContentLoss']:>14,.0f}  I=${v_data[slr]['InventoryLoss']:>12,.0f}")

all_slrs = sorted(set(a_data) | set(v_data))

# --- Excel styles ---
HEADER_FONT = Font(name="Arial", bold=True, color="FFFFFF", size=11)
HEADER_FILL = PatternFill("solid", start_color="305496")
TOTAL_FONT  = Font(name="Arial", bold=True, size=11)
TOTAL_FILL  = PatternFill("solid", start_color="D9E1F2")
BODY_FONT   = Font(name="Arial", size=10)
THIN        = Side(border_style="thin", color="BFBFBF")
BORDER      = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
CENTER      = Alignment(horizontal="center", vertical="center")
CURRENCY    = '"$"#,##0;("$"#,##0);"-"'
PCT         = "0.000%"

# --- Build the workbook ---
wb = Workbook()

# Sheet 1 + 2: one per coastal type (loop over both)
for sheet_idx, (sheet_name, data) in enumerate([("CoastalA", a_data), ("CoastalV", v_data)]):
    if sheet_idx == 0:
        ws = wb.active
        ws.title = sheet_name
    else:
        ws = wb.create_sheet(sheet_name)

    # Header row
    headers = ["SLR (in)", "Building Loss", "Content Loss", "Inventory Loss", "Total Loss", "Source File"]
    for col, h in enumerate(headers, start=1):
        c = ws.cell(row=1, column=col, value=h)
        c.font = HEADER_FONT; c.fill = HEADER_FILL; c.alignment = CENTER; c.border = BORDER

    # Data rows
    row = 2
    for slr in all_slrs:
        ws.cell(row=row, column=1, value=slr).alignment = CENTER
        if slr in data:
            d = data[slr]
            ws.cell(row=row, column=2, value=d["BldgLoss"]).number_format = CURRENCY
            ws.cell(row=row, column=3, value=d["ContentLoss"]).number_format = CURRENCY
            ws.cell(row=row, column=4, value=d["InventoryLoss"]).number_format = CURRENCY
            ws.cell(row=row, column=5, value=f"=SUM(B{row}:D{row})").number_format = CURRENCY
            ws.cell(row=row, column=6, value=d["file"])
        else:
            ws.cell(row=row, column=6, value="(no file)")
        for col in range(1, 7):
            cell = ws.cell(row=row, column=col)
            if not cell.font or not cell.font.bold:
                cell.font = BODY_FONT
            cell.border = BORDER
        row += 1

    # Grand total
    last = row - 1
    ws.cell(row=row, column=1, value="ALL SLR (sum)").alignment = CENTER
    for letter in "BCDE":
        ci = "ABCDEF".index(letter) + 1
        c = ws.cell(row=row, column=ci, value=f"=SUM({letter}2:{letter}{last})")
        c.number_format = CURRENCY
    for col in range(1, 7):
        cell = ws.cell(row=row, column=col)
        cell.font = TOTAL_FONT; cell.fill = TOTAL_FILL; cell.border = BORDER

    # Column widths and freeze panes
    for i, w in enumerate([10, 18, 18, 18, 18, 60], start=1):
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.freeze_panes = "A2"

# Sheet 3: Comparison
ws = wb.create_sheet("Comparison")
headers = ["SLR (in)",
           "A: Bldg", "A: Content", "A: Inventory", "A: Total",
           "V: Bldg", "V: Content", "V: Inventory", "V: Total",
           "Diff (V − A)", "Diff %"]
for col, h in enumerate(headers, start=1):
    c = ws.cell(row=1, column=col, value=h)
    c.font = HEADER_FONT; c.fill = HEADER_FILL; c.alignment = CENTER; c.border = BORDER

row = 2
for slr in all_slrs:
    a = a_data.get(slr, {})
    v = v_data.get(slr, {})
    ws.cell(row=row, column=1, value=slr).alignment = CENTER
    ws.cell(row=row, column=2, value=a.get("BldgLoss", 0)).number_format = CURRENCY
    ws.cell(row=row, column=3, value=a.get("ContentLoss", 0)).number_format = CURRENCY
    ws.cell(row=row, column=4, value=a.get("InventoryLoss", 0)).number_format = CURRENCY
    ws.cell(row=row, column=5, value=f"=SUM(B{row}:D{row})").number_format = CURRENCY
    ws.cell(row=row, column=6, value=v.get("BldgLoss", 0)).number_format = CURRENCY
    ws.cell(row=row, column=7, value=v.get("ContentLoss", 0)).number_format = CURRENCY
    ws.cell(row=row, column=8, value=v.get("InventoryLoss", 0)).number_format = CURRENCY
    ws.cell(row=row, column=9, value=f"=SUM(F{row}:H{row})").number_format = CURRENCY
    ws.cell(row=row, column=10, value=f"=I{row}-E{row}").number_format = CURRENCY
    ws.cell(row=row, column=11, value=f"=IFERROR((I{row}-E{row})/E{row},0)").number_format = PCT
    for col in range(1, 12):
        cell = ws.cell(row=row, column=col)
        if col != 1: cell.font = BODY_FONT
        cell.border = BORDER
    row += 1

# Comparison grand total
last = row - 1
ws.cell(row=row, column=1, value="ALL SLR (sum)").alignment = CENTER
for letter in "BCDEFGHIJ":
    ci = "ABCDEFGHIJK".index(letter) + 1
    c = ws.cell(row=row, column=ci, value=f"=SUM({letter}2:{letter}{last})")
    c.number_format = CURRENCY
ws.cell(row=row, column=11, value=f"=IFERROR((I{row}-E{row})/E{row},0)").number_format = PCT
for col in range(1, 12):
    cell = ws.cell(row=row, column=col)
    cell.font = TOTAL_FONT; cell.fill = TOTAL_FILL; cell.border = BORDER

for i, w in enumerate([10, 14, 14, 14, 16, 14, 14, 14, 16, 16, 10], start=1):
    ws.column_dimensions[get_column_letter(i)].width = w
ws.freeze_panes = "B2"

wb.save(out_xlsx)
print(f"\nSaved: {out_xlsx}")

Reading CoastalA folder...
  CoastalA  SLR=108: B=$   375,549,848  C=$   709,591,876  I=$  19,680,153
  CoastalA  SLR=12: B=$     1,496,051  C=$     3,845,385  I=$     498,565
  CoastalA  SLR=36: B=$    24,547,049  C=$    33,499,212  I=$     891,564
  CoastalA  SLR=52: B=$    74,920,098  C=$   117,216,726  I=$   2,901,099
  CoastalA  SLR=77: B=$   200,008,768  C=$   356,926,635  I=$   9,765,777
  CoastalA  SLR=96: B=$   303,778,783  C=$   560,978,503  I=$  15,696,375
  CoastalA  SLR=12: B=$             0  C=$             0  I=$           0

Reading CoastalV folder...
  CoastalV  SLR=108: B=$   375,558,904  C=$   709,596,404  I=$  19,680,153
  CoastalV  SLR=12: B=$     1,496,051  C=$     3,845,385  I=$     498,565
  CoastalV  SLR=36: B=$    24,547,049  C=$    33,499,212  I=$     891,564
  CoastalV  SLR=52: B=$    74,920,098  C=$   117,216,726  I=$   2,901,099
  CoastalV  SLR=77: B=$   199,993,637  C=$   356,926,635  I=$   9,765,777
  CoastalV  SLR=96: B=$   303,787,840  C=$   560,982,09

In [3]:
# ============================================================
# CELL 3 — Building-by-building diff CSVs (only buildings that differ between A and V)
# ============================================================
out_dir = a_dir.parent / "QA_AvV_Differences"
out_dir.mkdir(exist_ok=True)

# Columns to compare row-by-row
COMPARE_COLS = ["BldgDmgPct", "BldgLossUSD",
                "ContDmgPct", "ContentLossUSD",
                "InvDmgPct",  "InventoryLossUSD",
                "BDDF_ID", "CDDF_ID", "IDDF_ID"]

ID_COL = "UserDefinedFltyId"
CONTEXT_COLS = ["Occ", "NumStories", "FoundationType",
                "FirstFloorHt", "Cost", "Depth_in_Struc", "Latitude", "Longitude"]

summary_rows = []

for slr in EXPECTED_SLR:
    # Find the non-_sorted CSV for this SLR in each folder (inline, no helper)
    a_file = None
    for p in sorted(a_dir.glob("*.csv")):
        if "sorted" in p.name.lower():
            continue
        for n in re.findall(r"\d+", p.name):
            if int(n) == slr:
                a_file = p
                break
        if a_file:
            break

    v_file = None
    for p in sorted(v_dir.glob("*.csv")):
        if "sorted" in p.name.lower():
            continue
        for n in re.findall(r"\d+", p.name):
            if int(n) == slr:
                v_file = p
                break
        if v_file:
            break

    if not a_file or not v_file:
        print(f"SLR {slr}: missing file — A: {a_file}  V: {v_file}")
        continue

    a = pd.read_csv(a_file, low_memory=False).set_index(ID_COL).sort_index()
    v = pd.read_csv(v_file, low_memory=False).set_index(ID_COL).sort_index()

    # Only keep columns that exist in both
    cmp_cols = [c for c in COMPARE_COLS  if c in a.columns and c in v.columns]
    ctx_cols = [c for c in CONTEXT_COLS  if c in a.columns]

    # Build a True/False mask: True where any comparison column differs
    diff_mask = pd.Series(False, index=a.index)
    for col in cmp_cols:
        if pd.api.types.is_numeric_dtype(a[col]):
            diff_mask |= ((a[col] - v[col]).abs() > 0.01)
        else:
            diff_mask |= (a[col].astype(str) != v[col].astype(str))

    n_total = len(a)
    n_diff  = int(diff_mask.sum())
    n_same  = n_total - n_diff
    print(f"SLR {slr:3d}: {n_diff:>4} buildings differ, {n_same:>6} same  (of {n_total:,} total)")

    if n_diff == 0:
        summary_rows.append({"SLR_in": slr, "n_total": n_total, "n_diff": 0, "n_same": n_same,
                             "A_total_BldgLoss": a["BldgLossUSD"].sum(),
                             "V_total_BldgLoss": v["BldgLossUSD"].sum()})
        continue

    # Build side-by-side comparison for differing rows
    diff_ids = diff_mask[diff_mask].index
    out = pd.DataFrame(index=diff_ids)
    for c in ctx_cols:
        out[c] = a.loc[diff_ids, c]
    for c in cmp_cols:
        out[f"A_{c}"] = a.loc[diff_ids, c]
        out[f"V_{c}"] = v.loc[diff_ids, c]
        if pd.api.types.is_numeric_dtype(a[c]):
            out[f"diff_{c}"] = v.loc[diff_ids, c] - a.loc[diff_ids, c]

    out_path = out_dir / f"QA_AvV_diff_SLR_{slr}in.csv"
    out.to_csv(out_path)
    print(f"           wrote: {out_path.name}")

    summary_rows.append({
        "SLR_in": slr, "n_total": n_total, "n_diff": n_diff, "n_same": n_same,
        "A_total_BldgLoss": a["BldgLossUSD"].sum(),
        "V_total_BldgLoss": v["BldgLossUSD"].sum(),
        "diff_BldgLoss_VminusA": v["BldgLossUSD"].sum() - a["BldgLossUSD"].sum(),
    })

# Save summary
summary = pd.DataFrame(summary_rows)
summary_path = out_dir / "AvV_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"\nSummary written: {summary_path}")
print(summary.to_string(index=False))

SLR  12:    0 buildings differ,  99198 same  (of 99,198 total)
SLR  36:    0 buildings differ,  99198 same  (of 99,198 total)
SLR 48: missing file — A: None  V: None
SLR  52:    0 buildings differ,  99198 same  (of 99,198 total)
SLR  77:    8 buildings differ,  99190 same  (of 99,198 total)
           wrote: QA_AvV_diff_SLR_77in.csv
SLR 84: missing file — A: None  V: None
SLR  96:    8 buildings differ,  99190 same  (of 99,198 total)
           wrote: QA_AvV_diff_SLR_96in.csv
SLR 108:    8 buildings differ,  99190 same  (of 99,198 total)
           wrote: QA_AvV_diff_SLR_108in.csv

Summary written: C:\Users\MeaganBrown\Pathways Dropbox\Projects\202501.Solano.Bayshore\400-Technical\450-FAST\QA_AvV_Differences\AvV_summary.csv
 SLR_in  n_total  n_diff  n_same  A_total_BldgLoss  V_total_BldgLoss  diff_BldgLoss_VminusA
     12    99198       0   99198      1.496051e+06      1.496051e+06                    NaN
     36    99198       0   99198      2.454705e+07      2.454705e+07              

In [4]:
print("=== CoastalA files and what SLR they match ===")
for p in sorted(a_dir.glob("*.csv")):
    if "sorted" in p.name.lower():
        continue
    matched = None
    for n in re.findall(r"\d+", p.name):
        if int(n) in EXPECTED_SLR:
            matched = int(n)
            break
    print(f"  {p.name}  ->  matched SLR {matched}")

print("\n=== CoastalV files and what SLR they match ===")
for p in sorted(v_dir.glob("*.csv")):
    if "sorted" in p.name.lower():
        continue
    matched = None
    for n in re.findall(r"\d+", p.name):
        if int(n) in EXPECTED_SLR:
            matched = int(n)
            break
    print(f"  {p.name}  ->  matched SLR {matched}")

=== CoastalA files and what SLR they match ===
  NSI_Solano_FAST_Solano_inundation_rast_108.csv  ->  matched SLR 108
  NSI_Solano_FAST_Solano_inundation_rast_12.csv  ->  matched SLR 12
  NSI_Solano_FAST_Solano_inundation_rast_36.csv  ->  matched SLR 36
  NSI_Solano_FAST_Solano_inundation_rast_52.csv  ->  matched SLR 52
  NSI_Solano_FAST_Solano_inundation_rast_77.csv  ->  matched SLR 77
  NSI_Solano_FAST_Solano_inundation_rast_96.csv  ->  matched SLR 96
  QA_NSI_Solano_FAST_Solano_inundation_rast_12.csv  ->  matched SLR 12

=== CoastalV files and what SLR they match ===
  NSI_Solano_FAST_Solano_inundation_rast_108.csv  ->  matched SLR 108
  NSI_Solano_FAST_Solano_inundation_rast_12.csv  ->  matched SLR 12
  NSI_Solano_FAST_Solano_inundation_rast_36.csv  ->  matched SLR 36
  NSI_Solano_FAST_Solano_inundation_rast_52.csv  ->  matched SLR 52
  NSI_Solano_FAST_Solano_inundation_rast_77.csv  ->  matched SLR 77
  NSI_Solano_FAST_Solano_inundation_rast_96.csv  ->  matched SLR 96
